# December 2024 Pilot Data Acquisition and Inspection

## Milestone 1 Objective: 
# This project analyzes the historical reliability of nonstop flights between JFK and LAX during the Christmas travel season. We will study whether scheduled flights arrive on time, experience meaningful or severe delays, are cancelled, or are diverted, and later compare these outcomes across operating airlines, directions, scheduled departure times, and holiday travel phases.

# We begin with the complete December 2024 BTS dataset as a pilot month. Before downloading the remaining nine monthly files, we need to confirm that the official dataset contains the required fields, includes sufficient JFK–LAX and LAX–JFK records, and has data quality that supports the approved analysis.

# Each row represents one scheduled nonstop flight segment, not one aircraft. This pilot inspection will help us understand the dataset’s structure, missing values, route volume, and likely cleaning requirements.

In [2]:
import pandas as pd

In [3]:
from pathlib import Path
Path.cwd()

PosixPath('/Users/prajaktakadukar/Documents/flight-delay-airline-reliability/notebooks')

In [4]:
notebook_dir = Path.cwd()
project_root = notebook_dir.parent 

data_path = (
    project_root
    / "data"
    / "raw"
    / "2024-12"
    / "T_ONTIME_REPORTING.csv"
)
data_path

PosixPath('/Users/prajaktakadukar/Documents/flight-delay-airline-reliability/data/raw/2024-12/T_ONTIME_REPORTING.csv')

In [5]:
data_path.exists()

True

In [6]:
preview_df = pd.read_csv(data_path, nrows=5)

In [7]:
type(preview_df)

pandas.DataFrame

In [8]:
preview_df.shape

(5, 37)

In [9]:
preview_df.columns

Index(['YEAR', 'MONTH', 'DAY_OF_MONTH', 'DAY_OF_WEEK', 'FL_DATE',
       'OP_UNIQUE_CARRIER', 'OP_CARRIER_AIRLINE_ID', 'OP_CARRIER', 'TAIL_NUM',
       'OP_CARRIER_FL_NUM', 'ORIGIN_AIRPORT_ID', 'ORIGIN', 'DEST_AIRPORT_ID',
       'DEST', 'CRS_DEP_TIME', 'DEP_TIME', 'DEP_DELAY', 'WHEELS_OFF',
       'WHEELS_ON', 'CRS_ARR_TIME', 'ARR_TIME', 'ARR_DELAY', 'ARR_DEL15',
       'CANCELLED', 'CANCELLATION_CODE', 'DIVERTED', 'CRS_ELAPSED_TIME',
       'ACTUAL_ELAPSED_TIME', 'AIR_TIME', 'DISTANCE', 'CARRIER_DELAY',
       'WEATHER_DELAY', 'NAS_DELAY', 'SECURITY_DELAY', 'LATE_AIRCRAFT_DELAY',
       'DIV_REACHED_DEST', 'DIV_ARR_DELAY'],
      dtype='str')

In [10]:
preview_df[
    [
        "FL_DATE",
        "OP_UNIQUE_CARRIER",
        "ORIGIN",
        "DEST",
        "CRS_DEP_TIME",
        "ARR_DELAY",
        "CANCELLED",
        "DIVERTED",
    ]
]

,FL_DATE,OP_UNIQUE_CARRIER,ORIGIN,DEST,CRS_DEP_TIME,ARR_DELAY,CANCELLED,DIVERTED
0,12/1/2024 12:00:00 AM,9E,CLT,LGA,1444,37.0,0.0,0.0
1,12/1/2024 12:00:00 AM,9E,LGA,DSM,1730,56.0,0.0,0.0
2,12/1/2024 12:00:00 AM,9E,JFK,CLT,1159,17.0,0.0,0.0
3,12/1/2024 12:00:00 AM,9E,JFK,RDU,700,15.0,0.0,0.0
4,12/1/2024 12:00:00 AM,9E,RDU,JFK,929,26.0,0.0,0.0


In [11]:
flights_df = pd.read_csv(data_path)

/var/folders/hx/g52p7s1d201_1w5kv2sqfl_00000gn/T/ipykernel_10092/3888969872.py:1: DtypeWarning: Columns (0: CANCELLATION_CODE) have mixed types. Specify dtype option on import or set low_memory=False.
  flights_df = pd.read_csv(data_path)


Pandas examined CANCELLATION_CODE and could not confidently infer one consistent data type while reading the large CSV in chunks. A DtypeWarning occurs when read_csv() encounters non-uniform type inference within a column.
For this field, we expect:
cancelled flights may contain letter codes;
most non-cancelled flights may have missing values.
That likely explains the warning, but we will inspect the column before deciding. We will not blindly suppress it.


In [12]:
flights_df.shape

(590581, 37)

### Initial load result

The full December 2024 dataset contains 590,581 rows and 37 columns. Pandas loaded the file successfully, but issued a mixed-type warning for `CANCELLATION_CODE`, which needs to be investigated before changing the import settings.

### Cancellation fields

`CANCELLED` is a binary indicator: 1 means the flight was cancelled and 0 means it was not cancelled.

`CANCELLATION_CODE` records the reported cancellation category:
- A: Air Carrier
- B: Extreme Weather
- C: National Aviation System
- D: Security

A missing cancellation code is expected for flights that were not cancelled.

Inspecting the cancellation code

In [13]:
flights_df["CANCELLATION_CODE"].dtype

<StringDtype(storage='python', na_value=nan)>

In [14]:
flights_df["CANCELLATION_CODE"].value_counts(dropna=False)

CANCELLATION_CODE
NaN    586430
B        3122
A         737
C         292
Name: count, dtype: int64

In [15]:
flights_df["CANCELLED"].value_counts(dropna=False)

CANCELLED
0.0    586430
1.0      4151
Name: count, dtype: int64

### Validation of cancellation fields

The dataset contains two related cancellation fields:

- `CANCELLED` is a binary indicator where `1` means the flight was cancelled and `0` means it was not cancelled.
- `CANCELLATION_CODE` records the reported reason for cancellation:
  - `A`: Air Carrier
  - `B`: Extreme Weather
  - `C`: National Aviation System
  - `D`: Security

Most values in `CANCELLATION_CODE` are missing because the field does not apply to flights that were not cancelled. This is expected missingness rather than automatically indicating poor data quality.

The dataset contains 4,151 flights with `CANCELLED = 1`, and there are also 4,151 non-missing cancellation codes. Although these totals match, matching totals alone do not prove that every individual row is consistent.

The following row-level contradictions must therefore be checked:

1. A flight has `CANCELLED = 1`, but its cancellation code is missing.
2. A flight has `CANCELLED = 0`, but it has a cancellation code.

#A flight is marked as cancelled, but has a cancellation code.

In [16]:
cancelled_missing_code = (
    (flights_df["CANCELLED"] == 1)
    & flights_df["CANCELLATION_CODE"].isna()
)

In [17]:
cancelled_missing_code.shape

(590581,)

In [18]:
cancelled_missing_code.sum()

np.int64(0)

### A flight is marked as not cancelled, but still has a cancellation code.

In [19]:
not_cancelled_with_code = (
    (flights_df["CANCELLED"] == 0)
    & flights_df["CANCELLATION_CODE"].notna()
)

In [20]:
not_cancelled_with_code.sum()

np.int64(0)

#### Cancellation consistency result

The row-level validation found no contradictions between `CANCELLED` and `CANCELLATION_CODE`.

- Zero cancelled flights were missing a cancellation code.
- Zero non-cancelled flights had a cancellation code.

The missing values in `CANCELLATION_CODE` therefore appear to be expected: the field does not apply to flights that were not cancelled.

In [21]:
flights_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 590581 entries, 0 to 590580
Data columns (total 37 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   YEAR                   590581 non-null  int64  
 1   MONTH                  590581 non-null  int64  
 2   DAY_OF_MONTH           590581 non-null  int64  
 3   DAY_OF_WEEK            590581 non-null  int64  
 4   FL_DATE                590581 non-null  str    
 5   OP_UNIQUE_CARRIER      590581 non-null  str    
 6   OP_CARRIER_AIRLINE_ID  590581 non-null  int64  
 7   OP_CARRIER             590581 non-null  str    
 8   TAIL_NUM               589960 non-null  str    
 9   OP_CARRIER_FL_NUM      590581 non-null  int64  
 10  ORIGIN_AIRPORT_ID      590581 non-null  int64  
 11  ORIGIN                 590581 non-null  str    
 12  DEST_AIRPORT_ID        590581 non-null  int64  
 13  DEST                   590581 non-null  str    
 14  CRS_DEP_TIME           590581 non-null  int64  

In [22]:
missing_counts = flights_df.isna().sum()

In [23]:
missing_counts.shape

(37,)

In [24]:
 (missing_counts > 0).sum()

np.int64(18)

### Displaying only columns with missing values

In [25]:
missing_counts[missing_counts > 0].sort_values(ascending=False)

DIV_ARR_DELAY          589233
DIV_REACHED_DEST       588983
CANCELLATION_CODE      586430
LATE_AIRCRAFT_DELAY    466860
SECURITY_DELAY         466860
NAS_DELAY              466860
WEATHER_DELAY          466860
CARRIER_DELAY          466860
AIR_TIME                 5749
ACTUAL_ELAPSED_TIME      5749
ARR_DEL15                5749
ARR_DELAY                5749
ARR_TIME                 4401
WHEELS_ON                4401
WHEELS_OFF               4038
DEP_DELAY                3916
DEP_TIME                 3885
TAIL_NUM                  621
dtype: int64

### Which rows are both cancelled and missing an actual departure time?

In [26]:
cancelled_missing_dep_time = (
    (flights_df["CANCELLED"] == 1)
    & flights_df["DEP_TIME"].isna()
)

In [27]:
pd.crosstab(
    flights_df["CANCELLED"],
    flights_df["DEP_TIME"].isna(),
    rownames=["CANCELLED"],
    colnames=["DEP_TIME_MISSING"]
)

DEP_TIME_MISSING,False,True
CANCELLED,,
0.0,586430,0
1.0,266,3885


In [28]:
type(flights_df["CANCELLED"] == 1)

pandas.Series

#### Cancellation and departure-time relationship

Of the 4,151 cancelled flights:

- 3,885 have a missing `DEP_TIME`, suggesting that no actual departure time was recorded.
- 266 have a recorded `DEP_TIME`.

All 586,430 non-cancelled flights have a recorded departure time. Therefore, missing `DEP_TIME` is fully associated with cancelled flights in this pilot dataset, but not every cancelled flight is missing `DEP_TIME`. The 266 cancelled flights with recorded departure times require further inspection before interpretation.

### The initial assumption that every cancelled flight must have a missing `DEP_TIME` was incorrect. BTS departure time represents gate departure, not necessarily takeoff. Of the 4,151 cancelled flights, 266 have a recorded departure time, meaning some departure-related activity occurred before cancellation. These records require further inspection using fields such as `WHEELS_OFF` before their operational sequence can be interpreted.

In [29]:
cancelled_with_dep_time = flights_df[
    (flights_df["CANCELLED"] == 1)
    & flights_df["DEP_TIME"].notna()
]

In [30]:
cancelled_with_dep_time.shape

(266, 37)

In [31]:
cancelled_with_dep_time["WHEELS_OFF"].notna().value_counts()

WHEELS_OFF
False    153
True     113
Name: count, dtype: int64

#### Takeoff status among cancelled flights with departure times

Among the 266 cancelled flights with a recorded `DEP_TIME`:

- 153 have no recorded `WHEELS_OFF` time.
- 113 have a recorded `WHEELS_OFF` time.

The first group may have left the gate without taking off. The second group appears to have become airborne before ultimately being classified as cancelled. These records require further inspection before assigning an operational explanation.

In [32]:
#Now isolate the 113 rows:

cancelled_with_takeoff = cancelled_with_dep_time[
    cancelled_with_dep_time["WHEELS_OFF"].notna()
]

In [33]:
cancelled_with_takeoff.shape

(113, 37)

In [34]:
cancelled_with_takeoff[
    ["WHEELS_ON", "ARR_TIME", "AIR_TIME"]
].notna().sum()

WHEELS_ON    0
ARR_TIME     0
AIR_TIME     0
dtype: int64

#### Cancelled records with `WHEELS_OFF`

Among the 113 cancelled flights with a recorded `WHEELS_OFF` value:

- none has a recorded `WHEELS_ON`;
- none has a recorded `ARR_TIME`;
- none has a recorded `AIR_TIME`.

Therefore, a populated `WHEELS_OFF` value alone is insufficient to conclude that these flights completed an airborne segment. These records contain partial or unusual operational information and require closer inspection before interpretation.

In [35]:
cancelled_with_takeoff[
    [
        "FL_DATE",
        "OP_UNIQUE_CARRIER",
        "OP_CARRIER_FL_NUM",
        "ORIGIN",
        "DEST",
        "DEP_TIME",
        "WHEELS_OFF",
        "WHEELS_ON",
        "ARR_TIME",
        "AIR_TIME",
        "CANCELLATION_CODE",
    ]
].head(10)

,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER_FL_NUM,ORIGIN,DEST,DEP_TIME,WHEELS_OFF,WHEELS_ON,ARR_TIME,AIR_TIME,CANCELLATION_CODE
12454,12/1/2024 12:00:00 AM,OO,4141,SEA,RDM,1728.0,1742.0,NaN,NaN,NaN,B
12784,12/1/2024 12:00:00 AM,OO,3419,SEA,BLI,2300.0,2317.0,NaN,NaN,NaN,B
13898,12/1/2024 12:00:00 AM,OO,5425,SFO,RDM,1710.0,1726.0,NaN,NaN,NaN,B
25566,12/2/2024 12:00:00 AM,AS,89,ANC,ADQ,1304.0,1314.0,NaN,NaN,NaN,B
30504,12/2/2024 12:00:00 AM,G4,282,AZA,BLI,1915.0,1929.0,NaN,NaN,NaN,B
30517,12/2/2024 12:00:00 AM,G4,96,LAS,EUG,1103.0,1116.0,NaN,NaN,NaN,B
52621,12/3/2024 12:00:00 AM,OO,5777,DEN,BOI,1737.0,1759.0,NaN,NaN,NaN,B
70280,12/4/2024 12:00:00 AM,OO,5425,SFO,RDM,1649.0,1711.0,NaN,NaN,NaN,B
70367,12/4/2024 12:00:00 AM,OO,5380,SFO,SBP,1837.0,1853.0,NaN,NaN,NaN,B
70576,12/4/2024 12:00:00 AM,OO,3324,SAN,RDM,1543.0,1614.0,NaN,NaN,NaN,B


### These records contain a reported wheels-off time but no reported landing, arrival, or air-time information. The available fields are incomplete for reconstructing the full operational sequence.

In [36]:
cancelled_with_takeoff["CANCELLATION_CODE"].value_counts(dropna=False) #value_counts() checked all 113 rows

CANCELLATION_CODE
B    98
A    13
C     2
Name: count, dtype: int64

#### Cancellation categories among records with `WHEELS_OFF`

Among the 113 cancelled flights with a recorded `WHEELS_OFF` value:

- 98 have cancellation code `B`;
- 13 have cancellation code `A`;
- 2 have cancellation code `C`.

The first ten displayed records all had code `B`, but that preview was not representative of the entire filtered group. Checking all 113 rows showed that multiple cancellation categories were present.

### Confirm that the downloaded file really contains only December 2024 records.

#### Year validation

All 590,581 records have `YEAR = 2024`. No records from another year and no missing year values were found.

In [37]:
flights_df["YEAR"].value_counts(dropna=False)

YEAR
2024    590581
Name: count, dtype: int64

#### Month validation

All 590,581 records have `MONTH = 12`, confirming that the file contains December records only. No missing or unexpected month values were found.

In [38]:
flights_df["MONTH"].value_counts(dropna=False)

MONTH
12    590581
Name: count, dtype: int64

#### Flight-date format

`FL_DATE` was imported as text in the format `month/day/year 12:00:00 AM`. The midnight component is part of the stored calendar-date representation and is not the flight's departure time.

In [39]:
flights_df["FL_DATE"].dtype

<StringDtype(storage='python', na_value=nan)>

In [40]:
flights_df["FL_DATE"].head()

0    12/1/2024 12:00:00 AM
1    12/1/2024 12:00:00 AM
2    12/1/2024 12:00:00 AM
3    12/1/2024 12:00:00 AM
4    12/1/2024 12:00:00 AM
Name: FL_DATE, dtype: str

#### Flight-date format

`FL_DATE` was imported as text in the format `month/day/year 12:00:00 AM`. The midnight component is part of the stored calendar-date representation and is not the flight's departure time.

### Create a temporary parsed-date Series
We need pandas to understand these values as dates so that we can validate the earliest date, latest date, and number of days represented.

In [41]:
parsed_flight_dates = pd.to_datetime(
    flights_df["FL_DATE"],
    format="%m/%d/%Y %I:%M:%S %p"
)

### The format symbols mean:
%m — month number
%d — day of month
%Y — four-digit year
%I — hour using a 12-hour clock
%M — minutes
%S — seconds
%p — AM or PM

In [42]:
parsed_flight_dates.dtype

dtype('<M8[us]')

In [43]:
parsed_flight_dates.min() #earliest date 

Timestamp('2024-12-01 00:00:00')

In [44]:
parsed_flight_dates.max() #latest date

Timestamp('2024-12-31 00:00:00')

### To verify that all 31 calendar dates appear, rather than only checking the endpoints.


In [45]:
parsed_flight_dates.nunique()

31

#### Date-range validation

After temporarily converting `FL_DATE` from text to a datetime type:

- the earliest date is December 1, 2024;
- the latest date is December 31, 2024;
- all 31 distinct December dates are represented.

The original `FL_DATE` column remains unchanged.

In [46]:
# To validate whether the separate DAY_OF_MONTH column agrees with the day extracted from FL_DATE.

day_mismatch = (
    flights_df["DAY_OF_MONTH"]
    != parsed_flight_dates.dt.day
)

In [47]:
day_mismatch.sum()

np.int64(0)

In [48]:
cancelled_flight_data = flights_df[
    ["OP_CARRIER", "ORIGIN", "DEST", "CANCELLED"]
]

In [49]:
cancelled_flight_data.head()

,OP_CARRIER,ORIGIN,DEST,CANCELLED
0,9E,CLT,LGA,0.0
1,9E,LGA,DSM,0.0
2,9E,JFK,CLT,0.0
3,9E,JFK,RDU,0.0
4,9E,RDU,JFK,0.0


In [50]:
#cancelled_flight_data.shape

In [51]:
#type(cancelled_flight_data)

In [52]:
#type(cancelled_flight_data["CANCELLED"])

#### Validation of `DAY_OF_MONTH` against `FL_DATE`

The `DAY_OF_MONTH` column was compared with the day number extracted from the parsed `FL_DATE`.

The comparison used `!=`, which means **“is not equal to.”**

For example:

- `12 != 12` returns `False` because the values match.
- `12 != 13` returns `True` because the values do not match.

Therefore, a `True` value in `day_mismatch` would represent an inconsistency between `DAY_OF_MONTH` and `FL_DATE`.

The sum of `day_mismatch` was `0`, meaning that all 590,581 rows have a `DAY_OF_MONTH` value that agrees with the date stored in `FL_DATE`.

In [53]:
day_mismatch = (
    flights_df["DAY_OF_MONTH"]
    != parsed_flight_dates.dt.day
)

In [54]:
day_mismatch.sum()

np.int64(0)

In [55]:
flights_df["DAY_OF_WEEK"].value_counts().sort_index()

DAY_OF_WEEK
1    101060
2     85899
3     69713
4     79599
5     80624
6     71414
7    102272
Name: count, dtype: int64

In [56]:
parsed_flight_dates.dt.day_name().head()

0    Sunday
1    Sunday
2    Sunday
3    Sunday
4    Sunday
Name: FL_DATE, dtype: str

In [57]:
flights_df["DAY_OF_WEEK"].head()

0    7
1    7
2    7
3    7
4    7
Name: DAY_OF_WEEK, dtype: int64

In [58]:
expected_day_code = parsed_flight_dates.dt.dayofweek + 1

In [59]:
weekday_mismatch = (
    flights_df["DAY_OF_WEEK"] != expected_day_code
)

weekday_mismatch.sum()

np.int64(0)

#### Validation of `DAY_OF_WEEK` against `FL_DATE`

The weekday code stored in `DAY_OF_WEEK` was compared with the weekday calculated from the parsed `FL_DATE`.

Pandas numbers weekdays from 0 to 6, where Monday = 0 and Sunday = 6. BTS uses the same ordering shifted by 1, so the expected BTS weekday code was calculated by adding 1.

The mismatch count was 0, meaning all 590,581 rows have a `DAY_OF_WEEK` value consistent with their calendar date.

In [60]:
flights_df[["ORIGIN", "DEST"]].head()

,ORIGIN,DEST
0,CLT,LGA
1,LGA,DSM
2,JFK,CLT
3,JFK,RDU
4,RDU,JFK


In [61]:
jfk_to_lax = flights_df[(flights_df["ORIGIN"] == "JFK") & (flights_df["DEST"]== "LAX")]

In [62]:
jfk_to_lax.shape

(875, 37)

In [63]:
lax_to_jfk = flights_df[(flights_df["ORIGIN"] == "LAX") & (flights_df["DEST"]== "JFK")]

In [64]:
lax_to_jfk.shape

(873, 37)

In [65]:
total_jfk_lax = jfk_to_lax.shape[0] + lax_to_jfk.shape[0]

In [66]:
total_jfk_lax

1748

### Practice Section from here onwards

In [67]:
cancelled_fl = flights_df[flights_df["CANCELLED"]==1]

In [68]:
cancelled_fl.shape

(4151, 37)

In [69]:
flight_not_cancelled = flights_df[flights_df["CANCELLED"]==0]

In [70]:
flight_not_cancelled.shape

(586430, 37)

In [71]:
flights_df[["ORIGIN", "DEST"]]

,ORIGIN,DEST
0,CLT,LGA
1,LGA,DSM
2,JFK,CLT
3,JFK,RDU
4,RDU,JFK
...,...,...
590576,BUF,LGA
590577,GRR,LGA
590578,LGA,GRR
590579,LGA,PWM


In [72]:
flights_df["DEP_DELAY"]

0         29.0
1         59.0
2         21.0
3         13.0
4         26.0
          ... 
590576    -9.0
590577    -5.0
590578    -2.0
590579    -5.0
590580    -1.0
Name: DEP_DELAY, Length: 590581, dtype: float64

In [73]:
dep_delay60 = flights_df[flights_df["DEP_DELAY"]>60.0]

In [74]:
dep_delay60.shape

(43519, 37)

In [75]:
dep_delay60nc = flights_df[(flights_df["DEP_DELAY"]>60 )& (flights_df["CANCELLED"]==0)]

In [76]:
dep_delay60nc.shape

(43425, 37)

In [77]:
samp_df = flights_df[(flights_df["ORIGIN"]=="PHX")|(flights_df["DEP_DELAY"]>60)]

In [78]:
samp_df.shape

(59862, 37)

In [79]:
both_df = flights_df[(flights_df["ORIGIN"]=="PHX")&(flights_df["DEP_DELAY"]>60)]

In [80]:
both_df.shape

(999, 37)

In [81]:
AA_operated = flights_df[flights_df["OP_CARRIER"] == "AA"]

In [82]:
AA_operated.shape

(77737, 37)

In [83]:
AA_PHX = flights_df[(flights_df["OP_CARRIER"] == "AA") & (flights_df["ORIGIN"] == "PHX")]

In [84]:
AA_PHX.shape

(4531, 37)

### Find American Airlines flights that either departed from PHX or arrived at PHX.

In [85]:
arr_or_dep_phx = flights_df[(flights_df["OP_CARRIER"] == "AA") & ((flights_df["ORIGIN"] == "PHX") | (flights_df["DEST"] == "PHX"))]

In [86]:
arr_or_dep_phx.shape

(9060, 37)

### Finds all the Airlines present inside the dataset

In [87]:
flights_df["OP_CARRIER"].unique()

<StringArray>
['9E', 'AA', 'AS', 'B6', 'DL', 'F9', 'G4', 'HA', 'MQ', 'NK', 'OH', 'OO', 'UA',
 'WN', 'YX']
Length: 15, dtype: str

### it returned a NumPy Array

In [88]:
flights_df["OP_CARRIER"].nunique()

15

### .unique()    # What are the unique values?
### .nunique()   # How many unique values are there?

In [89]:
#Finding all distinct destination airports.
flights_df["DEST"].unique()

<StringArray>
['LGA', 'DSM', 'CLT', 'RDU', 'JFK', 'ATL', 'TYS', 'CHO', 'CSG', 'XNA',
 ...
 'GUM', 'SPN', 'SCC', 'CKB', 'SMX', 'PPG', 'ADK', 'BIH', 'EAU', 'MGW']
Length: 337, dtype: str

In [90]:
#Finding the number of distinct destination airports.
flights_df["DEST"].nunique()

337

In [91]:
#Finding all distinct cancellation codes.
flights_df["CANCELLATION_CODE"].unique()

<StringArray>
[nan, 'A', 'B', 'C']
Length: 4, dtype: str

In [92]:
flights_df["CANCELLATION_CODE"].nunique()
# shows 3 because nunique() excludes missing values by default.

3

In [93]:
flights_df["OP_CARRIER"].value_counts()

OP_CARRIER
WN    116276
DL     83283
AA     77737
OO     66601
UA     64030
YX     27279
MQ     22584
OH     20903
B6     20771
AS     20027
NK     18662
9E     17351
F9     17294
G4     11084
HA      6699
Name: count, dtype: int64

In [94]:
#How many flights originated from each airport?
flights_df["ORIGIN"].value_counts()

ORIGIN
ATL    29014
DEN    25905
DFW    25814
ORD    22668
CLT    17832
       ...  
PPG       12
CKB       11
LSE        9
SMX        9
ADK        8
Name: count, Length: 337, dtype: int64

### Among only the cancelled flights, how many cancellations came from each origin airport?

"Among cancelled flights, count each origin"
        ↓
filter first → value_counts()

In [95]:
#step 1: filter the cancelled flights
cancelled_flights = flights_df[flights_df["CANCELLED"]==1]
#step 2: count cancellations by origin
cancelled_by_origin = cancelled_flights["ORIGIN"].value_counts()

In [96]:
cancelled_by_origin

ORIGIN
DFW    536
SAN    206
BOS    166
ORD    134
IAH    113
      ... 
MEI      1
CWA      1
PSG      1
FWA      1
HTS      1
Name: count, Length: 268, dtype: int64

In [97]:
missing_dep_delay = flights_df[flights_df["DEP_DELAY"].isna()]

In [98]:
#How many flights have missing departure delay?
missing_dep_delay.shape

(3916, 37)

### DEP_DELAY is missing AND the flight was cancelled

In [99]:
dep_delay_cancelled = flights_df[(flights_df["DEP_DELAY"].isna()) & (flights_df['CANCELLED'] == 1)]

In [100]:
dep_delay_cancelled.shape

(3916, 37)

In [101]:
flights_df[flights_df["DEP_DELAY"].isna()]

,YEAR,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER_AIRLINE_ID,OP_CARRIER,TAIL_NUM,OP_CARRIER_FL_NUM,...,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,DIV_REACHED_DEST,DIV_ARR_DELAY
1092,2024,12,1,7,12/1/2024 12:00:00 AM,AA,19805,AA,N191UW,2241,...,NaN,NaN,600.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3989,2024,12,1,7,12/1/2024 12:00:00 AM,AS,19930,AS,N594AS,73,...,NaN,NaN,95.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4002,2024,12,1,7,12/1/2024 12:00:00 AM,AS,19930,AS,N609AS,32,...,NaN,NaN,894.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4018,2024,12,1,7,12/1/2024 12:00:00 AM,AS,19930,AS,N613AS,66,...,NaN,NaN,198.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4037,2024,12,1,7,12/1/2024 12:00:00 AM,AS,19930,AS,N622AS,64,...,NaN,NaN,82.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
588656,2024,12,31,2,12/31/2024 12:00:00 AM,WN,19393,WN,N8697C,1186,...,NaN,NaN,1164.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
588657,2024,12,31,2,12/31/2024 12:00:00 AM,WN,19393,WN,N8697C,2693,...,NaN,NaN,813.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
589604,2024,12,31,2,12/31/2024 12:00:00 AM,WN,19393,WN,N8933Q,1282,...,NaN,NaN,722.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
589712,2024,12,31,2,12/31/2024 12:00:00 AM,WN,19393,WN,N925WN,1252,...,NaN,NaN,296.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [102]:
#flights_df["DEP_DELAY"].isna().value_counts()

In [103]:
non_missing_dep_delay = flights_df[
    flights_df["DEP_DELAY"].notna()
]

non_missing_carrier = non_missing_dep_delay["OP_CARRIER"].value_counts()

In [104]:
non_missing_carrier

OP_CARRIER
WN    115737
DL     83226
AA     76988
OO     65765
UA     63722
YX     27120
MQ     22295
OH     20688
B6     20552
AS     19913
NK     18570
9E     17292
F9     17185
G4     10947
HA      6665
Name: count, dtype: int64

In [105]:
missing_dep_delay = flights_df[flights_df["DEP_DELAY"].isna()]

flights_missing_dep_delay = missing_dep_delay["OP_CARRIER"].value_counts()

In [106]:
flights_missing_dep_delay 

OP_CARRIER
OO    836
AA    749
WN    539
UA    308
MQ    289
B6    219
OH    215
YX    159
G4    137
AS    114
F9    109
NK     92
9E     59
DL     57
HA     34
Name: count, dtype: int64

In [107]:
flights_operated_AA = flights_df[flights_df["OP_CARRIER"] == "AA"]

origin_from_airport = flights_operated_AA["ORIGIN"].value_counts()

In [108]:
origin_from_airport

ORIGIN
DFW    13622
CLT     9965
MIA     5349
PHX     4531
ORD     3890
       ...  
MSO       12
BTV        9
LEX        8
AVL        4
HSV        4
Name: count, Length: 123, dtype: int64

In [109]:
dep_delay_present = flights_df[flights_df["DEP_DELAY"].notna()]

In [110]:
dep_delay_present

,YEAR,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER_AIRLINE_ID,OP_CARRIER,TAIL_NUM,OP_CARRIER_FL_NUM,...,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,DIV_REACHED_DEST,DIV_ARR_DELAY
0,2024,12,1,7,12/1/2024 12:00:00 AM,9E,20363,9E,N131EV,4804,...,128.0,77.0,544.0,0.0,0.0,8.0,0.0,29.0,NaN,NaN
1,2024,12,1,7,12/1/2024 12:00:00 AM,9E,20363,9E,N131EV,4904,...,189.0,152.0,1031.0,22.0,0.0,0.0,0.0,34.0,NaN,NaN
2,2024,12,1,7,12/1/2024 12:00:00 AM,9E,20363,9E,N131EV,5070,...,121.0,90.0,541.0,0.0,0.0,0.0,0.0,17.0,NaN,NaN
3,2024,12,1,7,12/1/2024 12:00:00 AM,9E,20363,9E,N131EV,5097,...,111.0,75.0,427.0,13.0,0.0,2.0,0.0,0.0,NaN,NaN
4,2024,12,1,7,12/1/2024 12:00:00 AM,9E,20363,9E,N131EV,5294,...,100.0,69.0,427.0,11.0,0.0,0.0,0.0,15.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
590576,2024,12,31,2,12/31/2024 12:00:00 AM,YX,20452,YX,N880RW,5825,...,80.0,52.0,292.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
590577,2024,12,31,2,12/31/2024 12:00:00 AM,YX,20452,YX,N882RW,5754,...,119.0,90.0,618.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
590578,2024,12,31,2,12/31/2024 12:00:00 AM,YX,20452,YX,N882RW,5754,...,123.0,98.0,618.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
590579,2024,12,31,2,12/31/2024 12:00:00 AM,YX,20452,YX,N882RW,5848,...,76.0,47.0,269.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [111]:
aa_missing_dep_delay = flights_df[(flights_df["OP_CARRIER"]== "AA") & (flights_df["DEP_DELAY"].isna())]

In [112]:
aa_missing_dep_delay.shape

(749, 37)

In [113]:
aa_flights = flights_df[flights_df["OP_CARRIER"] == "AA"]

aa_flights.sort_values("DEP_DELAY", ascending=False)

,YEAR,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER_AIRLINE_ID,OP_CARRIER,TAIL_NUM,OP_CARRIER_FL_NUM,...,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,DIV_REACHED_DEST,DIV_ARR_DELAY
495975,2024,12,27,5,12/27/2024 12:00:00 AM,AA,19805,AA,N866NN,2158,...,216.0,194.0,1781.0,3234.0,0.0,0.0,0.0,7.0,NaN,NaN
342628,2024,12,19,4,12/19/2024 12:00:00 AM,AA,19805,AA,N649AW,1561,...,127.0,114.0,931.0,3208.0,0.0,0.0,0.0,0.0,NaN,NaN
535628,2024,12,29,7,12/29/2024 12:00:00 AM,AA,19805,AA,N835AW,1760,...,144.0,130.0,872.0,2495.0,0.0,0.0,0.0,0.0,NaN,NaN
211788,2024,12,12,4,12/12/2024 12:00:00 AM,AA,19805,AA,N977UY,2981,...,148.0,124.0,813.0,2455.0,0.0,0.0,0.0,0.0,NaN,NaN
342097,2024,12,19,4,12/19/2024 12:00:00 AM,AA,19805,AA,N335RT,756,...,130.0,108.0,802.0,2427.0,0.0,0.0,0.0,0.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
536509,2024,12,29,7,12/29/2024 12:00:00 AM,AA,19805,AA,N977UY,815,...,NaN,NaN,599.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
536521,2024,12,29,7,12/29/2024 12:00:00 AM,AA,19805,AA,N979NN,2222,...,NaN,NaN,930.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
556120,2024,12,30,1,12/30/2024 12:00:00 AM,AA,19805,AA,N9018E,1883,...,NaN,NaN,1047.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
556121,2024,12,30,1,12/30/2024 12:00:00 AM,AA,19805,AA,N9018E,1883,...,NaN,NaN,1047.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [114]:
wn_nonmi_depdelay = flights_df[(flights_df["OP_CARRIER"]=="WN") & (flights_df["DEP_DELAY"].notna())]



wn_sorted = wn_nonmi_depdelay.sort_values("DEP_DELAY", ascending=False)

In [115]:
wn_sorted.shape

(115737, 37)

In [116]:
not_missing_depdelay = flights_df[
    flights_df["DEP_DELAY"].notna()
]

sorted_flights = not_missing_depdelay.sort_values(
    ["OP_CARRIER", "DEP_DELAY"],
    ascending=[True, False]
)

In [117]:
sorted_flights.shape

(586665, 37)

In [118]:
cancelled_flights = flights_df[
    flights_df["CANCELLED"] == 1
]

cancelled_sorted = cancelled_flights.sort_values(
    ["CANCELLATION_CODE", "DEP_DELAY"],
    ascending=[True, False]
)

In [119]:
cancelled_sorted.head(10)

,YEAR,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER_AIRLINE_ID,OP_CARRIER,TAIL_NUM,OP_CARRIER_FL_NUM,...,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,DIV_REACHED_DEST,DIV_ARR_DELAY
390783,2024,12,21,6,12/21/2024 12:00:00 AM,MQ,20398,MQ,N313BS,3575,...,NaN,NaN,1258.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
362838,2024,12,20,5,12/20/2024 12:00:00 AM,AA,19805,AA,N465AN,2466,...,NaN,NaN,338.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
543598,2024,12,29,7,12/29/2024 12:00:00 AM,OH,20397,OH,N500AE,5445,...,NaN,NaN,411.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
222816,2024,12,12,4,12/12/2024 12:00:00 AM,UA,19977,UA,N439UA,1599,...,NaN,NaN,284.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
442546,2024,12,24,2,12/24/2024 12:00:00 AM,AA,19805,AA,N110UW,1323,...,NaN,NaN,925.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
526927,2024,12,28,6,12/28/2024 12:00:00 AM,UA,19977,UA,N17245,1780,...,NaN,NaN,966.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
405388,2024,12,22,7,12/22/2024 12:00:00 AM,B6,20409,B6,N2039J,2839,...,NaN,NaN,1576.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
257205,2024,12,14,6,12/14/2024 12:00:00 AM,OH,20397,OH,N632NN,5317,...,NaN,NaN,226.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
527264,2024,12,28,6,12/28/2024 12:00:00 AM,UA,19977,UA,N27526,2662,...,NaN,NaN,854.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
322705,2024,12,18,3,12/18/2024 12:00:00 AM,9E,20363,9E,N319PQ,5209,...,NaN,NaN,378.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [120]:
flights_df.groupby("OP_CARRIER")["DEP_DELAY"].mean().sort_values(ascending=False)

OP_CARRIER
G4    23.251302
OH    20.748502
B6    20.362009
AA    18.172365
OO    15.324093
NK    14.457351
F9    13.776666
WN    12.716798
MQ    11.558017
DL    11.270504
9E    10.926845
UA    10.852720
AS     9.311304
YX     6.156895
HA     6.019355
Name: DEP_DELAY, dtype: float64

In [121]:
flights_df.groupby("ORIGIN")["DEP_DELAY"].mean().sort_values(ascending=False)

ORIGIN
HGR    106.857143
JST     90.774194
MGW     72.621622
PBG     70.109375
ABR     62.101695
          ...    
YAK     -6.649123
PPG     -6.666667
EKO     -8.935484
SCC    -10.000000
ADK    -19.375000
Name: DEP_DELAY, Length: 337, dtype: float64

In [122]:
flights_df["OP_CARRIER"].value_counts()

OP_CARRIER
WN    116276
DL     83283
AA     77737
OO     66601
UA     64030
YX     27279
MQ     22584
OH     20903
B6     20771
AS     20027
NK     18662
9E     17351
F9     17294
G4     11084
HA      6699
Name: count, dtype: int64

In [123]:
flights_df.groupby("OP_CARRIER")["DEP_DELAY"].max().sort_values(ascending=False)

OP_CARRIER
AA    3274.0
MQ    1929.0
OH    1918.0
OO    1746.0
G4    1478.0
B6    1459.0
UA    1420.0
9E    1279.0
DL    1175.0
F9    1164.0
YX    1152.0
HA    1122.0
NK     979.0
WN     877.0
AS     544.0
Name: DEP_DELAY, dtype: float64

In [124]:
flights_df.groupby("OP_CARRIER")["DEP_DELAY"].min().sort_values(ascending=True)

OP_CARRIER
DL   -59.0
AS   -42.0
B6   -38.0
G4   -38.0
OH   -38.0
F9   -31.0
UA   -31.0
AA   -30.0
OO   -29.0
9E   -28.0
NK   -28.0
MQ   -24.0
YX   -24.0
HA   -22.0
WN   -17.0
Name: DEP_DELAY, dtype: float64

In [125]:
flights_df.groupby("OP_CARRIER")["DEP_DELAY"].agg(
    ["mean", "min", "max"]
)

,mean,min,max
OP_CARRIER,,,
9E,10.926845,-28.0,1279.0
AA,18.172365,-30.0,3274.0
AS,9.311304,-42.0,544.0
B6,20.362009,-38.0,1459.0
DL,11.270504,-59.0,1175.0
F9,13.776666,-31.0,1164.0
G4,23.251302,-38.0,1478.0
HA,6.019355,-22.0,1122.0
MQ,11.558017,-24.0,1929.0


In [126]:
three_airlines = flights_df[
    (flights_df["OP_CARRIER"] == "AA") |
    (flights_df["OP_CARRIER"] == "DL") |
    (flights_df["OP_CARRIER"] == "WN")
]

In [127]:
#three_airlines.head(10)

In [128]:
three_airlines.groupby("OP_CARRIER")["DEP_DELAY"].agg(["mean", "min", "max"])

,mean,min,max
OP_CARRIER,,,
AA,18.172365,-30.0,3274.0
DL,11.270504,-59.0,1175.0
WN,12.716798,-17.0,877.0


In [129]:
flights_df.isna().sum()

YEAR                          0
MONTH                         0
DAY_OF_MONTH                  0
DAY_OF_WEEK                   0
FL_DATE                       0
OP_UNIQUE_CARRIER             0
OP_CARRIER_AIRLINE_ID         0
OP_CARRIER                    0
TAIL_NUM                    621
OP_CARRIER_FL_NUM             0
ORIGIN_AIRPORT_ID             0
ORIGIN                        0
DEST_AIRPORT_ID               0
DEST                          0
CRS_DEP_TIME                  0
DEP_TIME                   3885
DEP_DELAY                  3916
WHEELS_OFF                 4038
WHEELS_ON                  4401
CRS_ARR_TIME                  0
ARR_TIME                   4401
ARR_DELAY                  5749
ARR_DEL15                  5749
CANCELLED                     0
CANCELLATION_CODE        586430
DIVERTED                      0
CRS_ELAPSED_TIME              0
ACTUAL_ELAPSED_TIME        5749
AIR_TIME                   5749
DISTANCE                      0
CARRIER_DELAY            466860
WEATHER_

In [130]:
flights_df.head(7)

,YEAR,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER_AIRLINE_ID,OP_CARRIER,TAIL_NUM,OP_CARRIER_FL_NUM,...,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,DIV_REACHED_DEST,DIV_ARR_DELAY
0,2024,12,1,7,12/1/2024 12:00:00 AM,9E,20363,9E,N131EV,4804,...,128.0,77.0,544.0,0.0,0.0,8.0,0.0,29.0,NaN,NaN
1,2024,12,1,7,12/1/2024 12:00:00 AM,9E,20363,9E,N131EV,4904,...,189.0,152.0,1031.0,22.0,0.0,0.0,0.0,34.0,NaN,NaN
2,2024,12,1,7,12/1/2024 12:00:00 AM,9E,20363,9E,N131EV,5070,...,121.0,90.0,541.0,0.0,0.0,0.0,0.0,17.0,NaN,NaN
3,2024,12,1,7,12/1/2024 12:00:00 AM,9E,20363,9E,N131EV,5097,...,111.0,75.0,427.0,13.0,0.0,2.0,0.0,0.0,NaN,NaN
4,2024,12,1,7,12/1/2024 12:00:00 AM,9E,20363,9E,N131EV,5294,...,100.0,69.0,427.0,11.0,0.0,0.0,0.0,15.0,NaN,NaN
5,2024,12,1,7,12/1/2024 12:00:00 AM,9E,20363,9E,N132EV,4806,...,111.0,79.0,457.0,0.0,0.0,6.0,0.0,77.0,NaN,NaN
6,2024,12,1,7,12/1/2024 12:00:00 AM,9E,20363,9E,N132EV,5203,...,53.0,30.0,152.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [131]:
flights_df.columns

Index(['YEAR', 'MONTH', 'DAY_OF_MONTH', 'DAY_OF_WEEK', 'FL_DATE',
       'OP_UNIQUE_CARRIER', 'OP_CARRIER_AIRLINE_ID', 'OP_CARRIER', 'TAIL_NUM',
       'OP_CARRIER_FL_NUM', 'ORIGIN_AIRPORT_ID', 'ORIGIN', 'DEST_AIRPORT_ID',
       'DEST', 'CRS_DEP_TIME', 'DEP_TIME', 'DEP_DELAY', 'WHEELS_OFF',
       'WHEELS_ON', 'CRS_ARR_TIME', 'ARR_TIME', 'ARR_DELAY', 'ARR_DEL15',
       'CANCELLED', 'CANCELLATION_CODE', 'DIVERTED', 'CRS_ELAPSED_TIME',
       'ACTUAL_ELAPSED_TIME', 'AIR_TIME', 'DISTANCE', 'CARRIER_DELAY',
       'WEATHER_DELAY', 'NAS_DELAY', 'SECURITY_DELAY', 'LATE_AIRCRAFT_DELAY',
       'DIV_REACHED_DEST', 'DIV_ARR_DELAY'],
      dtype='str')

In [132]:
flights_df.duplicated().sum() 

np.int64(0)

In [133]:
flights_df[flights_df.duplicated(keep=False)]

,YEAR,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER_AIRLINE_ID,OP_CARRIER,TAIL_NUM,OP_CARRIER_FL_NUM,...,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,DIV_REACHED_DEST,DIV_ARR_DELAY


In [135]:
flights_df["OP_CARRIER"].value_counts()

OP_CARRIER
WN    116276
DL     83283
AA     77737
OO     66601
UA     64030
YX     27279
MQ     22584
OH     20903
B6     20771
AS     20027
NK     18662
9E     17351
F9     17294
G4     11084
HA      6699
Name: count, dtype: int64

In [137]:
flights_df["DEP_DELAY"].isna().sum()

np.int64(3916)

In [138]:
flights_df[flights_df["DEP_DELAY"].isna()]

,YEAR,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER_AIRLINE_ID,OP_CARRIER,TAIL_NUM,OP_CARRIER_FL_NUM,...,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,DIV_REACHED_DEST,DIV_ARR_DELAY
1092,2024,12,1,7,12/1/2024 12:00:00 AM,AA,19805,AA,N191UW,2241,...,NaN,NaN,600.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3989,2024,12,1,7,12/1/2024 12:00:00 AM,AS,19930,AS,N594AS,73,...,NaN,NaN,95.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4002,2024,12,1,7,12/1/2024 12:00:00 AM,AS,19930,AS,N609AS,32,...,NaN,NaN,894.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4018,2024,12,1,7,12/1/2024 12:00:00 AM,AS,19930,AS,N613AS,66,...,NaN,NaN,198.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4037,2024,12,1,7,12/1/2024 12:00:00 AM,AS,19930,AS,N622AS,64,...,NaN,NaN,82.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
588656,2024,12,31,2,12/31/2024 12:00:00 AM,WN,19393,WN,N8697C,1186,...,NaN,NaN,1164.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
588657,2024,12,31,2,12/31/2024 12:00:00 AM,WN,19393,WN,N8697C,2693,...,NaN,NaN,813.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
589604,2024,12,31,2,12/31/2024 12:00:00 AM,WN,19393,WN,N8933Q,1282,...,NaN,NaN,722.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
589712,2024,12,31,2,12/31/2024 12:00:00 AM,WN,19393,WN,N925WN,1252,...,NaN,NaN,296.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [139]:
flights_df[flights_df["DEP_DELAY"].notna()]

,YEAR,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER_AIRLINE_ID,OP_CARRIER,TAIL_NUM,OP_CARRIER_FL_NUM,...,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,DIV_REACHED_DEST,DIV_ARR_DELAY
0,2024,12,1,7,12/1/2024 12:00:00 AM,9E,20363,9E,N131EV,4804,...,128.0,77.0,544.0,0.0,0.0,8.0,0.0,29.0,NaN,NaN
1,2024,12,1,7,12/1/2024 12:00:00 AM,9E,20363,9E,N131EV,4904,...,189.0,152.0,1031.0,22.0,0.0,0.0,0.0,34.0,NaN,NaN
2,2024,12,1,7,12/1/2024 12:00:00 AM,9E,20363,9E,N131EV,5070,...,121.0,90.0,541.0,0.0,0.0,0.0,0.0,17.0,NaN,NaN
3,2024,12,1,7,12/1/2024 12:00:00 AM,9E,20363,9E,N131EV,5097,...,111.0,75.0,427.0,13.0,0.0,2.0,0.0,0.0,NaN,NaN
4,2024,12,1,7,12/1/2024 12:00:00 AM,9E,20363,9E,N131EV,5294,...,100.0,69.0,427.0,11.0,0.0,0.0,0.0,15.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
590576,2024,12,31,2,12/31/2024 12:00:00 AM,YX,20452,YX,N880RW,5825,...,80.0,52.0,292.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
590577,2024,12,31,2,12/31/2024 12:00:00 AM,YX,20452,YX,N882RW,5754,...,119.0,90.0,618.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
590578,2024,12,31,2,12/31/2024 12:00:00 AM,YX,20452,YX,N882RW,5754,...,123.0,98.0,618.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
590579,2024,12,31,2,12/31/2024 12:00:00 AM,YX,20452,YX,N882RW,5848,...,76.0,47.0,269.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [140]:
flights_df[(flights_df["ORIGIN"] == "PHX") & (flights_df["DEP_DELAY"] > 60)]

,YEAR,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER_AIRLINE_ID,OP_CARRIER,TAIL_NUM,OP_CARRIER_FL_NUM,...,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,DIV_REACHED_DEST,DIV_ARR_DELAY
860,2024,12,1,7,12/1/2024 12:00:00 AM,AA,19805,AA,N140AN,1742,...,224.0,200.0,1773.0,0.0,0.0,0.0,0.0,51.0,NaN,NaN
1236,2024,12,1,7,12/1/2024 12:00:00 AM,AA,19805,AA,N324SH,3262,...,61.0,41.0,255.0,0.0,0.0,0.0,0.0,55.0,NaN,NaN
1396,2024,12,1,7,12/1/2024 12:00:00 AM,AA,19805,AA,N402AN,663,...,371.0,355.0,2860.0,30.0,0.0,0.0,0.0,0.0,NaN,NaN
1433,2024,12,1,7,12/1/2024 12:00:00 AM,AA,19805,AA,N417AN,1967,...,261.0,241.0,2133.0,0.0,0.0,0.0,0.0,56.0,NaN,NaN
1533,2024,12,1,7,12/1/2024 12:00:00 AM,AA,19805,AA,N455AN,1450,...,294.0,258.0,2300.0,0.0,0.0,5.0,0.0,153.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
587616,2024,12,31,2,12/31/2024 12:00:00 AM,WN,19393,WN,N7827A,3784,...,81.0,69.0,369.0,0.0,0.0,54.0,0.0,0.0,NaN,NaN
589674,2024,12,31,2,12/31/2024 12:00:00 AM,WN,19393,WN,N915WN,3909,...,83.0,71.0,369.0,0.0,0.0,93.0,0.0,7.0,NaN,NaN
589678,2024,12,31,2,12/31/2024 12:00:00 AM,WN,19393,WN,N917WN,1730,...,103.0,93.0,833.0,0.0,0.0,0.0,0.0,216.0,NaN,NaN
589933,2024,12,31,2,12/31/2024 12:00:00 AM,WN,19393,WN,N968WN,2098,...,77.0,65.0,369.0,1.0,0.0,0.0,0.0,104.0,NaN,NaN


In [141]:
flights_df.groupby("OP_CARRIER")["DEP_DELAY"].mean().sort_values(ascending=False)

OP_CARRIER
G4    23.251302
OH    20.748502
B6    20.362009
AA    18.172365
OO    15.324093
NK    14.457351
F9    13.776666
WN    12.716798
MQ    11.558017
DL    11.270504
9E    10.926845
UA    10.852720
AS     9.311304
YX     6.156895
HA     6.019355
Name: DEP_DELAY, dtype: float64

In [142]:
flights_df["DEP_DELAY"].agg(["mean", "min", "max"])

mean      13.559404
min      -59.000000
max     3274.000000
Name: DEP_DELAY, dtype: float64

In [146]:
flights_df.groupby("OP_CARRIER")["DEP_DELAY"].agg(["mean", "min", "max"]).sort_values(by="mean", ascending=False)

,mean,min,max
OP_CARRIER,,,
G4,23.251302,-38.0,1478.0
OH,20.748502,-38.0,1918.0
B6,20.362009,-38.0,1459.0
AA,18.172365,-30.0,3274.0
OO,15.324093,-29.0,1746.0
NK,14.457351,-28.0,979.0
F9,13.776666,-31.0,1164.0
WN,12.716798,-17.0,877.0
MQ,11.558017,-24.0,1929.0


In [144]:
flights_df.groupby("OP_CARRIER")["DEP_DELAY"].mean().sort_values(ascending=False)

OP_CARRIER
G4    23.251302
OH    20.748502
B6    20.362009
AA    18.172365
OO    15.324093
NK    14.457351
F9    13.776666
WN    12.716798
MQ    11.558017
DL    11.270504
9E    10.926845
UA    10.852720
AS     9.311304
YX     6.156895
HA     6.019355
Name: DEP_DELAY, dtype: float64

In [147]:
flights_df[
    (flights_df["OP_CARRIER"] == "AA") |
    (flights_df["OP_CARRIER"] == "DL") |
    (flights_df["OP_CARRIER"] == "WN")
]

,YEAR,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER_AIRLINE_ID,OP_CARRIER,TAIL_NUM,OP_CARRIER_FL_NUM,...,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,DIV_REACHED_DEST,DIV_ARR_DELAY
635,2024,12,1,7,12/1/2024 12:00:00 AM,AA,19805,AA,N101NN,2,...,319.0,289.0,2475.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
636,2024,12,1,7,12/1/2024 12:00:00 AM,AA,19805,AA,N101NN,28,...,329.0,296.0,2475.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
637,2024,12,1,7,12/1/2024 12:00:00 AM,AA,19805,AA,N101NN,306,...,364.0,342.0,2475.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
638,2024,12,1,7,12/1/2024 12:00:00 AM,AA,19805,AA,N102NN,10,...,296.0,277.0,2475.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
639,2024,12,1,7,12/1/2024 12:00:00 AM,AA,19805,AA,N102NN,302,...,371.0,339.0,2475.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
589936,2024,12,31,2,12/31/2024 12:00:00 AM,WN,19393,WN,N968WN,4802,...,79.0,57.0,369.0,12.0,0.0,0.0,0.0,112.0,NaN,NaN
589937,2024,12,31,2,12/31/2024 12:00:00 AM,WN,19393,WN,N968WN,827,...,NaN,NaN,850.0,NaN,NaN,NaN,NaN,NaN,1.0,135.0
589938,2024,12,31,2,12/31/2024 12:00:00 AM,WN,19393,WN,N969WN,255,...,101.0,89.0,719.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
589939,2024,12,31,2,12/31/2024 12:00:00 AM,WN,19393,WN,N969WN,3029,...,168.0,148.0,1012.0,8.0,0.0,0.0,0.0,7.0,NaN,NaN


In [148]:
flights_df["OP_CARRIER"].isin(["AA", "DL", "WN"])

0         False
1         False
2         False
3         False
4         False
          ...  
590576    False
590577    False
590578    False
590579    False
590580    False
Name: OP_CARRIER, Length: 590581, dtype: bool

In [149]:
flights_df[
    flights_df["OP_CARRIER"].isin(["AA", "DL", "WN"])
]

,YEAR,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER_AIRLINE_ID,OP_CARRIER,TAIL_NUM,OP_CARRIER_FL_NUM,...,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,DIV_REACHED_DEST,DIV_ARR_DELAY
635,2024,12,1,7,12/1/2024 12:00:00 AM,AA,19805,AA,N101NN,2,...,319.0,289.0,2475.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
636,2024,12,1,7,12/1/2024 12:00:00 AM,AA,19805,AA,N101NN,28,...,329.0,296.0,2475.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
637,2024,12,1,7,12/1/2024 12:00:00 AM,AA,19805,AA,N101NN,306,...,364.0,342.0,2475.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
638,2024,12,1,7,12/1/2024 12:00:00 AM,AA,19805,AA,N102NN,10,...,296.0,277.0,2475.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
639,2024,12,1,7,12/1/2024 12:00:00 AM,AA,19805,AA,N102NN,302,...,371.0,339.0,2475.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
589936,2024,12,31,2,12/31/2024 12:00:00 AM,WN,19393,WN,N968WN,4802,...,79.0,57.0,369.0,12.0,0.0,0.0,0.0,112.0,NaN,NaN
589937,2024,12,31,2,12/31/2024 12:00:00 AM,WN,19393,WN,N968WN,827,...,NaN,NaN,850.0,NaN,NaN,NaN,NaN,NaN,1.0,135.0
589938,2024,12,31,2,12/31/2024 12:00:00 AM,WN,19393,WN,N969WN,255,...,101.0,89.0,719.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
589939,2024,12,31,2,12/31/2024 12:00:00 AM,WN,19393,WN,N969WN,3029,...,168.0,148.0,1012.0,8.0,0.0,0.0,0.0,7.0,NaN,NaN


In [150]:
flights_df[flights_df["OP_CARRIER"].isin(["AA", "UA", "B6"])]

,YEAR,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER_AIRLINE_ID,OP_CARRIER,TAIL_NUM,OP_CARRIER_FL_NUM,...,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,DIV_REACHED_DEST,DIV_ARR_DELAY
635,2024,12,1,7,12/1/2024 12:00:00 AM,AA,19805,AA,N101NN,2,...,319.0,289.0,2475.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
636,2024,12,1,7,12/1/2024 12:00:00 AM,AA,19805,AA,N101NN,28,...,329.0,296.0,2475.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
637,2024,12,1,7,12/1/2024 12:00:00 AM,AA,19805,AA,N101NN,306,...,364.0,342.0,2475.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
638,2024,12,1,7,12/1/2024 12:00:00 AM,AA,19805,AA,N102NN,10,...,296.0,277.0,2475.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
639,2024,12,1,7,12/1/2024 12:00:00 AM,AA,19805,AA,N102NN,302,...,371.0,339.0,2475.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
586528,2024,12,31,2,12/31/2024 12:00:00 AM,UA,19977,UA,N899UA,2147,...,162.0,102.0,723.0,21.0,0.0,33.0,0.0,0.0,NaN,NaN
586529,2024,12,31,2,12/31/2024 12:00:00 AM,UA,19977,UA,N899UA,2202,...,175.0,148.0,1185.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
586530,2024,12,31,2,12/31/2024 12:00:00 AM,UA,19977,UA,N899UA,224,...,196.0,168.0,1185.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
586531,2024,12,31,2,12/31/2024 12:00:00 AM,UA,19977,UA,N91007,218,...,466.0,437.0,4243.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [156]:
(
    flights_df[flights_df["OP_CARRIER"].isin(["AA", "UA", "B6"])]
    .groupby("OP_CARRIER")["DEP_DELAY"]
    .mean()
    .sort_values(ascending=False)
)

OP_CARRIER
B6    20.362009
AA    18.172365
UA    10.852720
Name: DEP_DELAY, dtype: float64

In [157]:
flights_df.groupby("OP_CARRIER")["DEP_DELAY"].mean()

OP_CARRIER
9E    10.926845
AA    18.172365
AS     9.311304
B6    20.362009
DL    11.270504
F9    13.776666
G4    23.251302
HA     6.019355
MQ    11.558017
NK    14.457351
OH    20.748502
OO    15.324093
UA    10.852720
WN    12.716798
YX     6.156895
Name: DEP_DELAY, dtype: float64